# A3.1 · Default-deny on the tool call

**Function A — Securing AI Architectures → Securing the Architecture — Runtime and the Gateway**  ·  *Security of AI*

Builds on **[A2.8 · An audit trail the workload cannot forge](https://spbreed.github.io/cyber-commons/lessons/A2.8.html)**.

| | |
|---|---|
| Tools used | OPA / Rego, SPIFFE/SPIRE |

## What this lesson is

**What it covers.** Evaluate the same call under allow-by-default and deny-by-default policy and compare what gets through.

**Why a security engineer needs it.** Allow-by-default authorization is defeated by any argument the model can be persuaded to produce. The control it builds is: policy evaluated per call on (identity, tool, arguments, resource), denying unless a rule permits.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Identity has already failed. Something untrusted is in the context and the agent has decided to call a tool. The tool call is the last place a decision can still be made on facts rather than intent — this SPIFFE ID, this tool, this resource, this verb — and a policy written one notch vaguer than that cannot express the distinction the attack turns on.

> **At CyberTravels.** The last place a decision about that refund rests on facts rather than on intent. Identity has already failed, an injected instruction is in the context, and the tool call is where CyberTravels can still say no. R1, R3.

## 2 · The framework

```
   untrusted text in context ---> agent decides to call a tool
                                             |
                                    +--------v---------+
                                    |  policy decision |
                                    |  DEFAULT: DENY   |
                                    +--------+---------+
                                             |
                        allow only on facts: identity, scope,
                        resource, provenance of the motivating span

   the last point where a decision rests on facts rather than on intent
```

**Mitigates: T2 Tool Misuse · T3 Privilege Compromise · T6 Intent Breaking.**

The tool call is the moment text becomes consequence. It is also the last point
where a decision can be made on **facts** — this identity, this tool, these
arguments, this resource — rather than on intent, which nobody can read.

**Start from the identity, and be very specific about it.** Not "the Workflow
Agent"; the SPIFFE ID A2.3 issued —
`spiffe://cybertravels.com/ns/prod/sa/workflow-agent` — and against it, an
entitlement written at full resolution: which tools, on which resources, with
which verbs. Everything else in this lesson is a consequence of writing the
entitlement down at that resolution. A policy phrased one notch vaguer cannot
express the distinction the attack turns on, and the vagueness is invisible
until it is exploited.

Default-deny then means the absence of a rule is a refusal. That sounds like a
detail and it is the entire control, because it changes what a mistake costs.
Under allow-by-default, a permission somebody forgot to restrict is available to
an attacker. Under deny-by-default, a permission somebody forgot to grant is a
broken feature — which someone reports on Monday morning, loudly, and which
harms nobody.

The policy takes four inputs and all four matter:

- **identity** — the attested workload identity, from A2.3
- **tool** — which capability
- **arguments** — the actual values, not the schema
- **resource** — which specific thing

Dropping the fourth is the most common weakening. `run_query` permitted for the
Workflow Agent is not the same as `run_query` permitted *on the bookings table*,
and A1.5 was the difference between those two sentences. The verb is the second
most common: `charge_card` entitled for payments is not `refund` entitled for
payments, and R1 is the entire distance between them.

This does not stop the agent being persuaded. It stops persuasion mattering,
which is a better place to stand.

> **What this control closes.**
>
> Stands on the edge every topology shares: `agent_runtime -> tools`. Persuasion still happens; it just stops reaching anything.

## 3 · The entitlement, at full resolution

In [ ]:
# One agent's entire entitlement. Not "the Workflow Agent may query" - this
# SPIFFE ID, these tools, these resources, these verbs. Anything not written
# here is refused, so the file is also the complete answer to "what can this
# agent do", which no amount of reading the code will give you.
ENTITLEMENTS = {
 "spiffe://cybertravels.com/ns/prod/sa/workflow-agent": {
   "run_query":   {"table:bookings":        {"SELECT", "UPDATE"}},
   "charge_card": {"payments:booking":      {"CHARGE"}},   # CHARGE, not REFUND
   "send_email":  {"domain:cybertravels.com": {"*"}},
 },
 "spiffe://cybertravels.com/ns/prod/sa/advisor-agent": {
   "run_query":   {"table:itineraries":     {"SELECT"}},
 },
}

WF = "spiffe://cybertravels.com/ns/prod/sa/workflow-agent"

def decide(identity, tool, resource, verb, default_deny=True):
    """Four inputs. No matching rule means refuse."""
    resources = ENTITLEMENTS.get(identity, {}).get(tool)
    if resources is None:
        return (False, "no entitlement for this identity+tool") if default_deny \
               else (True, "allowed by default")
    for res_prefix in sorted(resources):
        if resource.startswith(res_prefix):
            verbs = resources[res_prefix]
            if "*" in verbs or verb in verbs:
                return True, f"{tool} on {res_prefix} permits {verb}"
            return False, f"{verb} not permitted on {res_prefix} (only {sorted(verbs)})"
    return (False, "resource outside the entitlement") if default_deny \
           else (True, "allowed by default")

for tool, res in sorted((t, r) for t, rs in ENTITLEMENTS[WF].items() for r in rs):
    print(f"   {tool:12s}{res:26s}{sorted(ENTITLEMENTS[WF][tool][res])}")

## 4 · The same five calls, evaluated both ways

In [ ]:
CALLS = [
 (WF, "run_query",   "table:bookings",           "SELECT"),  # intended
 (WF, "run_query",   "table:customer_pii",       "SELECT"),  # A1.5, resource
 (WF, "charge_card", "payments:booking",         "REFUND"),  # R1, verb
 (WF, "send_email",  "domain:archive.evil.example", "*"),    # A1.3, exfiltration
 (WF, "drop_table",  "table:bookings",           "*"),       # tool never granted
]

for mode in (False, True):
    label = "DEFAULT-DENY" if mode else "allow-by-default"
    allowed = 0
    print(f"{label}:")
    for identity, tool, resource, verb in CALLS:
        ok, why = decide(identity, tool, resource, verb, default_deny=mode)
        allowed += ok
        print(f"   {tool:12s}{resource:30s}{verb:7s}"
              f"{'ALLOW' if ok else 'deny ':6s}{why}")
    print(f"   -> {allowed}/{len(CALLS)} permitted\n")

print("Only the first call should succeed. Under allow-by-default four do, and")
print("each one is a real risk from Chapter 1 walking through.")
print()
print("Row three is the one to sit with: same identity, same tool, same")
print("resource, refused on the VERB. An entitlement attached to the tool")
print("instead of the call cannot express that distinction at all - and the")
print("distance between CHARGE and REFUND is the whole of R1.")
assert sum(decide(*c, default_deny=True)[0] for c in CALLS) == 1
assert not decide(WF, "charge_card", "payments:booking", "REFUND")[0]

## What you just proved

The Workflow Agent's entitlement prints at full resolution — three tools, three resources, explicit verbs. Five tool calls are then evaluated twice: under allow-by-default four succeed, each one a Chapter 1 risk walking through; under default-deny only the intended call survives, including a refusal on the verb `REFUND` for an identity, tool and resource that are all otherwise permitted.

## Your turn

Take one tool policy you have and check whether it names the resource *and* the verb. If it grants `run_query` rather than `SELECT on these tables`, it cannot express the difference that A1.5 and R1 both turn on.

---

**Next → [A3.2 · Sandboxed execution](https://spbreed.github.io/cyber-commons/lessons/A3.2.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A3.1.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A3.1.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*